# 03_playlist_tracks

DML: bronze_playlist_tracks — Raw playlist-to-track mappings.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("playlist_tracks", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.col("playlist_id"), F.explode("items").alias("item"))
    .select(
        F.col("playlist_id"),
        F.col("item.track.id").alias("track_id"),
        F.col("item.track.name").alias("track_name"),
        F.to_timestamp("item.added_at").alias("added_at"),
        F.col("item.added_by.id").alias("added_by"),
        F.to_json(F.col("item")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_playlist_tracks")
print(f"bronze_playlist_tracks: {bronze.count()} rows written")